# Credit Scoring — Fase 3: Modelado Baseline

Entrenamos 4 modelos baseline con validación cruzada estratificada.  
Métricas: **AUC-ROC**, **KS Statistic**, **Gini**, **Brier Score**.  
Comparamos 3 estrategias de manejo de desbalanceo: sin balanceo, `class_weight='balanced'`, y SMOTE.

In [ ]:
import sys
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    brier_score_loss, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
sys.path.append('..')

DATA_PROC = Path('../data/processed')
MODELS = Path('../models/saved')
FIGURES = Path('../reports/figures')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Carga de datos

Usamos `train_raw` para modelos de árbol y `train_woe` para Logistic Regression (requiere variables ya linealizadas).

In [ ]:
train_raw = pd.read_csv(DATA_PROC / 'train_raw.csv')
val_raw   = pd.read_csv(DATA_PROC / 'val_raw.csv')
train_woe = pd.read_csv(DATA_PROC / 'train_woe.csv')
val_woe   = pd.read_csv(DATA_PROC / 'val_woe.csv')

X_train_raw, y_train = train_raw.drop('target', axis=1), train_raw['target']
X_val_raw,   y_val   = val_raw.drop('target', axis=1),   val_raw['target']
X_train_woe          = train_woe.drop('target', axis=1)
X_val_woe            = val_woe.drop('target', axis=1)

print(f'Train: {X_train_raw.shape} | Val: {X_val_raw.shape}')
print(f'Default rate train: {y_train.mean():.2%} | val: {y_val.mean():.2%}')

## 2. Funciones de evaluación

Definimos las métricas estándar de credit scoring:  
- **KS Statistic**: máxima separación entre distribuciones de buenos y malos pagadores  
- **Gini**: 2 × AUC − 1, mide poder discriminatorio  
- **Brier Score**: calibración de probabilidades (menor es mejor)

In [ ]:
def ks_statistic(y_true: pd.Series, y_prob: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))


def evaluate_model(model, X_train, y_train, X_val, y_val, name: str) -> dict:
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_val)[:, 1]

    auc   = roc_auc_score(y_val, y_prob)
    ks    = ks_statistic(y_val, y_prob)
    gini  = 2 * auc - 1
    brier = brier_score_loss(y_val, y_prob)

    return {'model': name, 'AUC-ROC': auc, 'KS': ks, 'Gini': gini, 'Brier': brier,
            '_model_obj': model, '_y_prob': y_prob}


def cv_score(model, X, y, name: str, cv: int = 5) -> dict:
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_validate(
        model, X, y, cv=skf,
        scoring={'auc': 'roc_auc', 'brier': 'neg_brier_score'},
        return_train_score=False, n_jobs=-1
    )
    return {
        'model': name,
        'CV AUC mean': scores['test_auc'].mean(),
        'CV AUC std':  scores['test_auc'].std(),
    }

print('Funciones de evaluación definidas.')

## 3. Modelos baseline — sin balanceo

Primera iteración: modelos con hiperparámetros por defecto para establecer el piso de performance.

In [ ]:
models_no_balance = {
    'LogReg (WoE)': (LogisticRegression(max_iter=1000, random_state=42), X_train_woe, X_val_woe),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, random_state=42), X_train_raw, X_val_raw),
    'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1), X_train_raw, X_val_raw),
    'Gradient Boost': (GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42), X_train_raw, X_val_raw),
}

results_no_balance = []
trained_models = {}

for name, (model, X_tr, X_v) in models_no_balance.items():
    r = evaluate_model(model, X_tr, y_train, X_v, y_val, name)
    trained_models[name] = {'model': r.pop('_model_obj'), 'y_prob': r.pop('_y_prob'),
                             'X_val': X_v}
    results_no_balance.append(r)
    print(f"{name:20s} | AUC={r['AUC-ROC']:.4f} | KS={r['KS']:.4f} | Gini={r['Gini']:.4f} | Brier={r['Brier']:.4f}")

## 4. Impacto del manejo de desbalanceo

Comparamos `class_weight='balanced'` y SMOTE en los mejores modelos para documentar si el desbalanceo (~6.7% default) afecta la performance.

In [ ]:
results_balanced = []

# class_weight='balanced'
models_balanced = {
    'LogReg balanced': (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42), X_train_woe, X_val_woe),
    'RF balanced':     (RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1), X_train_raw, X_val_raw),
    'GB balanced':     (GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42), X_train_raw, X_val_raw),
}

for name, (model, X_tr, X_v) in models_balanced.items():
    r = evaluate_model(model, X_tr, y_train, X_v, y_val, name)
    r.pop('_model_obj'); r.pop('_y_prob')
    results_balanced.append(r)
    print(f"{name:22s} | AUC={r['AUC-ROC']:.4f} | KS={r['KS']:.4f} | Gini={r['Gini']:.4f} | Brier={r['Brier']:.4f}")

# SMOTE
print('\n--- SMOTE ---')
smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])
smote_pipe.fit(X_train_raw, y_train)
y_prob_smote = smote_pipe.predict_proba(X_val_raw)[:, 1]
auc_smote = roc_auc_score(y_val, y_prob_smote)
ks_smote  = ks_statistic(y_val, y_prob_smote)
print(f"RF + SMOTE             | AUC={auc_smote:.4f} | KS={ks_smote:.4f} | Gini={2*auc_smote-1:.4f} | Brier={brier_score_loss(y_val, y_prob_smote):.4f}")

## 5. Cross-validation (5-fold estratificado)

Validamos con CV para confirmar que los resultados no son producto de un split afortunado.

In [ ]:
# Unimos train+val para el CV
X_cv_raw = pd.concat([X_train_raw, X_val_raw])
X_cv_woe = pd.concat([X_train_woe, X_val_woe])
y_cv     = pd.concat([y_train, y_val])

cv_models = {
    'LogReg (WoE)':  (LogisticRegression(max_iter=1000, random_state=42), X_cv_woe),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, random_state=42), X_cv_raw),
    'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1), X_cv_raw),
    'Gradient Boost':(GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42), X_cv_raw),
}

cv_results = []
for name, (model, X) in cv_models.items():
    r = cv_score(model, X, y_cv, name)
    cv_results.append(r)
    print(f"{name:20s} | CV AUC: {r['CV AUC mean']:.4f} ± {r['CV AUC std']:.4f}")

## 6. Curvas ROC y Precision-Recall

Las curvas ROC comparan el poder discriminatorio. La curva PR es más informativa con datasets desbalanceados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = ['steelblue', 'tomato', 'green', 'purple']

for (name, info), color in zip(trained_models.items(), colors_map):
    y_prob = info['y_prob']

    # ROC
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)

    # PR
    prec, rec, _ = precision_recall_curve(y_val, y_prob)
    axes[1].plot(rec, prec, label=name, color=color)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC'); axes[0].legend(fontsize=8)

axes[1].axhline(y_val.mean(), color='gray', linestyle='--', linewidth=0.8, label='Baseline')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall'); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / '10_roc_pr_curves.png', bbox_inches='tight')
plt.show()

## 7. Tabla comparativa y selección del modelo baseline

Comparamos todos los modelos baseline para elegir el punto de partida para la Fase 4.

In [ ]:
results_df = pd.DataFrame(results_no_balance).set_index('model')
results_df = results_df.sort_values('AUC-ROC', ascending=False)

display(results_df.round(4).style.background_gradient(cmap='Blues', subset=['AUC-ROC', 'KS', 'Gini'])
                                  .background_gradient(cmap='RdYlGn_r', subset=['Brier']))

## 8. Matriz de confusión con umbral optimizado

El umbral por defecto (0.5) es inadecuado con desbalanceo. Optimizamos por **KS** (maximizar separación de clases), que es el criterio estándar en credit scoring.

In [ ]:
def optimal_threshold_ks(y_true, y_prob) -> float:
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    idx = np.argmax(tpr - fpr)
    return float(thresholds[idx])


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, info) in enumerate(trained_models.items()):
    y_prob = info['y_prob']
    thresh = optimal_threshold_ks(y_val, y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_val, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{name}\n(umbral KS = {thresh:.3f})', fontsize=9)

plt.suptitle('Matrices de confusión con umbral óptimo (KS)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / '11_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 9. Guardar resultados y mejor modelo baseline

In [ ]:
# Mejor modelo baseline por AUC
best_name = results_df['AUC-ROC'].idxmax()
best_model = trained_models[best_name]['model']

joblib.dump(best_model, MODELS / 'baseline_champion.joblib')
print(f'Modelo baseline campeón: {best_name}')

# Registro de experimentos
experiments_path = MODELS / 'experiments.json'
experiments = json.loads(experiments_path.read_text()) if experiments_path.exists() else {}
experiments['baseline'] = {
    'champion': best_name,
    'results': results_df.round(4).to_dict(),
    'cv_results': cv_results,
}
experiments_path.write_text(json.dumps(experiments, indent=2))
print('Resultados guardados en models/saved/experiments.json')

## Resumen Fase 3

**Modelos entrenados:** Logistic Regression (WoE), Decision Tree, Random Forest, Gradient Boosting

**Hallazgos clave:**
- El desbalanceo (~6.7%) tiene impacto moderado — `class_weight='balanced'` mejora recall de default a costa de más falsos positivos
- SMOTE no supera significativamente a `class_weight='balanced'` en AUC
- El umbral óptimo por KS es consistentemente inferior a 0.5 en todos los modelos

**Próximo paso:** `04_modeling_advanced.ipynb` — XGBoost con Optuna, LightGBM, CatBoost